In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
wandb_key = secrets.get_secret("WANDB_API_KEY")
print("WANDB_API_KEY length:", len(wandb_key))

WANDB_API_KEY length: 86


In [3]:
import wandb
wandb.login(key=wandb_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import transformers
import re
from pathlib import Path
from sklearn.model_selection import GroupKFold
from torch.utils.data import Dataset, DataLoader
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

torch: 2.10.0+cu128 cuda: True
transformers: 5.0.0


In [5]:
_PREFIXES = [
    "Pick the best possible answer:",
    "Select the most accurate option:",
    "Determine the correct option:",
    "Identify the correct statement:",
    "Choose the correct answer:",
    "Which of the following is correct?",
]
_SUFFIXES = [
    "among the listed options.",
    "based on the given context.",
    "from the following choices.",
    "carefully.",
]
_PREFIX_RE = re.compile(r"^\s*(?:" + "|".join(re.escape(p) for p in _PREFIXES) + r")\s*")
_SUFFIX_RE = re.compile(r"\s*(?:" + "|".join(re.escape(s) for s in _SUFFIXES) + r")\s*$")
_WS_RE = re.compile(r"\s+")

def normalize_stem(text):
    if not isinstance(text, str):
        return text
    s = text
    while True:
        before = s
        s = _PREFIX_RE.sub("", s)
        s = _SUFFIX_RE.sub("", s)
        if s == before:
            break
    s = _WS_RE.sub(" ", s).strip()
    return s

In [6]:
comp_dir = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
train = pd.read_csv(comp_dir / "train.csv")
train["stem"] = train["prompt"].apply(normalize_stem)

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
train["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(gkf.split(train, groups=train["stem"])):
    train.loc[val_idx, "fold"] = fold_idx
print(train["fold"].value_counts().sort_index())
assert (train["fold"].value_counts() == 400).all(), "fold sizes do not match local"

fold
0    400
1    400
2    400
3    400
4    400
Name: count, dtype: int64


In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID = tokenizer.pad_token_id
print("vocab:", VOCAB_SIZE, "pad_id:", PAD_ID)

OPT_COLS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {c: i for i, c in enumerate(OPT_COLS)}

class MCQPairDataset(Dataset):
    def __init__(self, df, tokenizer, stem_len=48, opt_len=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.stem_len = stem_len
        self.opt_len = opt_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        s = self.tokenizer(row["stem"], max_length=self.stem_len, padding="max_length", truncation=True, return_tensors="pt")
        o = self.tokenizer([row[c] for c in OPT_COLS], max_length=self.opt_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "stem_ids": s["input_ids"].squeeze(0),
            "stem_mask": s["attention_mask"].squeeze(0),
            "opt_ids": o["input_ids"],
            "opt_mask": o["attention_mask"],
            "label": LABEL2IDX[row["answer"]],
        }

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocab: 50265 pad_id: 1


In [8]:
import torch.nn as nn
import torch.nn.functional as F

class TextCNNEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, channels, kernels, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, channels, k, padding=k // 2) for k in kernels])
        self.out_dim = channels * len(kernels)

    def forward(self, ids, mask):
        x = self.embedding(ids).transpose(1, 2)
        m = mask.unsqueeze(1).bool()
        feats = []
        for conv in self.convs:
            c = F.relu(conv(x))
            c = c.masked_fill(~m, -1e9)
            feats.append(c.max(dim=2).values)
        return torch.cat(feats, dim=1)

class MCQTextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, channels, kernels, pad_id, hidden, dropout):
        super().__init__()
        self.encoder = TextCNNEncoder(vocab_size, embed_dim, channels, kernels, pad_id)
        d = self.encoder.out_dim
        self.scorer = nn.Sequential(
            nn.Linear(4 * d, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1)
        )

    def forward(self, stem_ids, stem_mask, opt_ids, opt_mask):
        u = self.encoder(stem_ids, stem_mask)
        b, k, l = opt_ids.shape
        v = self.encoder(opt_ids.view(b * k, l), opt_mask.view(b * k, l)).view(b, k, -1)
        u = u.unsqueeze(1).expand(-1, k, -1)
        inter = torch.cat([u, v, (u - v).abs(), u * v], dim=-1)
        return self.scorer(inter).squeeze(-1)

In [9]:
def compute_map3(logits, labels):
    top3 = logits.argsort(dim=1, descending=True)[:, :3]
    ap = torch.zeros(len(labels))
    for k in range(3):
        match = (top3[:, k] == labels)
        ap += match.float() / (k + 1)
    return ap.mean().item()

In [10]:
import random

SEED = 42
STEM_LEN = 48
OPT_LEN = 128
BATCH_SIZE = 32
EPOCHS = 15
PATIENCE = 3
LR = 1e-3
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
EMBED_DIM = 200
CHANNELS = 128
KERNELS = [3, 5, 7]
HIDDEN = 256
DROPOUT = 0.3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("config set, seed", SEED)

config set, seed 42


In [11]:
# 5-fold cv training, textcnn from scratch, hard-subset eval
import time

fold_scores = []
fold_hard = []

for fold in range(N_FOLDS):
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    train_df = train[train["fold"] != fold].reset_index(drop=True)
    val_df = train[train["fold"] == fold].reset_index(drop=True)
    train_ds = MCQPairDataset(train_df, tokenizer, STEM_LEN, OPT_LEN)
    val_ds = MCQPairDataset(val_df, tokenizer, STEM_LEN, OPT_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = MCQTextCNN(VOCAB_SIZE, EMBED_DIM, CHANNELS, KERNELS, PAD_ID, HIDDEN, DROPOUT).to("cuda")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    opt_lens = np.array([[len(str(val_df.iloc[r][c])) for c in OPT_COLS] for r in range(len(val_df))])
    hard = opt_lens.argmax(1) != val_df["answer"].map(LABEL2IDX).values

    run = wandb.init(
        project="23f2001849-t22026",
        entity="23f2001849-dl-genai-project",
        name=f"m2cnn_fold{fold}",
        group="m2_textcnn_cv",
        tags=["m2", "textcnn", "workstream-b", "scratch", "cv", "hardeval"],
        config={
            "fold": fold, "stem_len": STEM_LEN, "opt_len": OPT_LEN, "batch_size": BATCH_SIZE,
            "epochs": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY, "embed_dim": EMBED_DIM,
            "channels": CHANNELS, "kernels": KERNELS, "hidden": HIDDEN, "dropout": DROPOUT,
            "patience": PATIENCE, "seed": SEED,
        },
    )

    best_val_map3 = -1.0
    best_hard = float("nan")
    bad_epochs = 0
    best_ckpt_path = Path(f"/kaggle/working/m2cnn_fold{fold}_best.pt")
    t_start = time.time()

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []
        for batch in train_loader:
            stem_ids = batch["stem_ids"].to("cuda")
            stem_mask = batch["stem_mask"].to("cuda")
            opt_ids = batch["opt_ids"].to("cuda")
            opt_mask = batch["opt_mask"].to("cuda")
            labels = batch["label"].to("cuda")
            optimizer.zero_grad()
            logits = model(stem_ids, stem_mask, opt_ids, opt_mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()
            train_losses.append(loss.item())
            wandb.log({"train/loss": loss.item()})

        model.eval()
        val_logits_all, val_labels_all = [], []
        with torch.no_grad():
            for batch in val_loader:
                stem_ids = batch["stem_ids"].to("cuda")
                stem_mask = batch["stem_mask"].to("cuda")
                opt_ids = batch["opt_ids"].to("cuda")
                opt_mask = batch["opt_mask"].to("cuda")
                labels = batch["label"].to("cuda")
                logits = model(stem_ids, stem_mask, opt_ids, opt_mask)
                val_logits_all.append(logits.cpu())
                val_labels_all.append(labels.cpu())
        val_logits = torch.cat(val_logits_all)
        val_labels = torch.cat(val_labels_all)
        val_map3 = compute_map3(val_logits, val_labels)
        val_map3_hard = compute_map3(val_logits[torch.from_numpy(hard)], val_labels[torch.from_numpy(hard)]) if hard.sum() else float("nan")
        train_loss = float(np.mean(train_losses))
        elapsed = time.time() - t_start
        print(f"fold {fold} epoch {epoch}: train_loss={train_loss:.4f} val_map3={val_map3:.4f} val_map3_hard={val_map3_hard:.4f} elapsed={elapsed:.1f}s")
        wandb.log({"epoch": epoch, "epoch/train_loss": train_loss, "epoch/val_map3": val_map3, "epoch/val_map3_hard": val_map3_hard})

        if val_map3 > best_val_map3:
            best_val_map3 = val_map3
            best_hard = val_map3_hard
            bad_epochs = 0
            torch.save({"model_state": model.state_dict(), "val_map3": val_map3, "epoch": epoch, "fold": fold}, best_ckpt_path)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"  early stop at epoch {epoch}")
                break

    wandb.run.summary["best_val_map3"] = best_val_map3
    wandb.run.summary["best_val_map3_hard"] = best_hard
    run.finish()
    fold_scores.append(best_val_map3)
    fold_hard.append(best_hard)
    print(f"fold {fold} best_val_map3={best_val_map3:.4f} best_hard={best_hard:.4f}")

print("\nper-fold full:", [round(s, 4) for s in fold_scores])
print("per-fold hard:", [round(s, 4) for s in fold_hard])
print(f"cv mean full={np.mean(fold_scores):.4f}  cv mean hard={np.nanmean(fold_hard):.4f}")

fold 0 epoch 0: train_loss=0.5842 val_map3=0.4087 val_map3_hard=0.2765 elapsed=5.1s
fold 0 epoch 1: train_loss=0.0116 val_map3=0.4704 val_map3_hard=0.3190 elapsed=7.4s
fold 0 epoch 2: train_loss=0.0019 val_map3=0.4271 val_map3_hard=0.3288 elapsed=9.8s
fold 0 epoch 3: train_loss=0.0001 val_map3=0.4146 val_map3_hard=0.3170 elapsed=12.0s
fold 0 epoch 4: train_loss=0.0001 val_map3=0.4129 val_map3_hard=0.3157 elapsed=14.3s
  early stop at epoch 4


epoch,▁▃▅▆█
epoch/train_loss,█▁▁▁▁
epoch/val_map3,▁█▃▂▁
epoch/val_map3_hard,▁▇█▆▆
train/loss,█▇▅▅▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_map3,0.47042
best_val_map3_hard,0.31895
epoch,4
epoch/train_loss,6e-05
epoch/val_map3,0.41292
epoch/val_map3_hard,0.31569


fold 0 best_val_map3=0.4704 best_hard=0.3190


fold 1 epoch 0: train_loss=0.6483 val_map3=0.5450 val_map3_hard=0.3752 elapsed=2.5s
fold 1 epoch 1: train_loss=0.0035 val_map3=0.5313 val_map3_hard=0.3703 elapsed=4.9s
fold 1 epoch 2: train_loss=0.0006 val_map3=0.5404 val_map3_hard=0.3891 elapsed=7.3s
fold 1 epoch 3: train_loss=0.0006 val_map3=0.5446 val_map3_hard=0.3947 elapsed=9.8s
  early stop at epoch 3


epoch,▁▃▆█
epoch/train_loss,█▁▁▁
epoch/val_map3,█▁▆█
epoch/val_map3_hard,▂▁▆█
train/loss,█▇▆▄▃▄▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_map3,0.545
best_val_map3_hard,0.37517
epoch,3
epoch/train_loss,0.00057
epoch/val_map3,0.54458
epoch/val_map3_hard,0.3947


...

epoch,▁▃▅▆█
epoch/train_loss,█▁▁▁▁
epoch/val_map3,▁█▄▄▄
epoch/val_map3_hard,█▁▁▁▁
train/loss,▇█▆▅▄▁▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_map3,0.5675
best_val_map3_hard,0.5423
epoch,4
epoch/train_loss,5e-05
epoch/val_map3,0.54667
epoch/val_map3_hard,0.5423


fold 3 best_val_map3=0.5675 best_hard=0.5423


fold 4 epoch 0: train_loss=0.6490 val_map3=0.7242 val_map3_hard=0.6970 elapsed=2.5s
fold 4 epoch 1: train_loss=0.0115 val_map3=0.6350 val_map3_hard=0.5874 elapsed=5.1s
fold 4 epoch 2: train_loss=0.0016 val_map3=0.6342 val_map3_hard=0.5748 elapsed=7.6s
fold 4 epoch 3: train_loss=0.0003 val_map3=0.6908 val_map3_hard=0.5941 elapsed=10.2s
  early stop at epoch 3


epoch,▁▃▆█
epoch/train_loss,█▁▁▁
epoch/val_map3,█▁▁▅
epoch/val_map3_hard,█▂▁▂
train/loss,██▇▅▅▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_map3,0.72417
best_val_map3_hard,0.69704
epoch,3
epoch/train_loss,0.00033
epoch/val_map3,0.69083
epoch/val_map3_hard,0.59407


fold 4 best_val_map3=0.7242 best_hard=0.6970

per-fold full: [0.4704, 0.545, 0.5437, 0.5675, 0.7242]
per-fold hard: [0.319, 0.3752, 0.4719, 0.5423, 0.697]
cv mean full=0.5702  cv mean hard=0.4811
